In [2]:
%pip install geopandas networkx fiona pyproj shapely

Note: you may need to restart the kernel to use updated packages.


In [1]:
import geopandas as gpd
import pandas as pd
import networkx as nx
from shapely.geometry import Point
from collections import Counter


# Testing why the feature count is different. 

In [2]:
# original file feature count test


roads_exploded_test = gpd.read_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/data_for_redundancy/hawaii_roads_exploded.gpkg",
    layer="roads_exploded"
)

print(len(roads_exploded_test))
print(roads_exploded_test.columns.tolist()[:20])
print(roads_exploded_test.crs)


589735
['full_id', 'osm_id', 'osm_type', 'highway', 'line', 'maxspeed:advisory:forward', 'taxi', 'crossing:markings:colour', 'crossing:continuous', 'phone', 'handrail:center', 'url', 'traffic_signals:vibration', 'access_aisle', 'ramp:wheelchair', 'postal_code', 'flashing_lights', 'destroyed:highway', 'damage:type', 'damage:event:wikidata']
EPSG:4326


In [3]:
#testing original exploded lines 

expected_segments = 0

for geom in roads_exploded_test.geometry:

    if geom is None or geom.is_empty: 
        continue

    if geom.geom_type == "LineString":
        expected_segments += len(geom.coords) - 1

    elif geom.geom_type == "MultiLineString": 
        for part in geom.geoms:
            expected_segments += len(part.coords) - 1

print ("Expected exploded segments:", expected_segments)

Expected exploded segments: 589735


# Uploading files 

In [8]:
roads_filtered = gpd.read_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/osm_roads_filtered.gpkg",
    layer="osm_roads_filtered"
)

print(len(roads_filtered))
print(roads_filtered.columns.tolist()[:20])
print(roads_filtered.crs)


82580
['full_id', 'osm_id', 'osm_type', 'highway', 'line', 'maxspeed:advisory:forward', 'taxi', 'crossing:markings:colour', 'crossing:continuous', 'phone', 'handrail:center', 'url', 'traffic_signals:vibration', 'access_aisle', 'ramp:wheelchair', 'postal_code', 'flashing_lights', 'destroyed:highway', 'damage:type', 'damage:event:wikidata']
EPSG:4326


In [10]:
roads_exploded = gpd.read_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/roads_exploded.gpkg",
    layer="roads_exploded"
)

print(len(roads_exploded))
print(roads_exploded.columns.tolist()[:20])
print(roads_exploded.crs)

590355
['full_id', 'osm_id', 'osm_type', 'highway', 'line', 'maxspeed:advisory:forward', 'taxi', 'crossing:markings:colour', 'crossing:continuous', 'phone', 'handrail:center', 'url', 'traffic_signals:vibration', 'access_aisle', 'ramp:wheelchair', 'postal_code', 'flashing_lights', 'destroyed:highway', 'damage:type', 'damage:event:wikidata']
EPSG:4326


In [ ]:
# Double checking that we get the right feature count for roads_exploded.

In [14]:
expected_segments = 0

for geom in roads_filtered.geometry:

    if geom is None or geom.is_empty: 
        continue

    if geom.geom_type == "LineString":
        expected_segments += len(geom.coords) - 1

    elif geom.geom_type == "MultiLineString": 
        for part in geom.geoms:
            expected_segments += len(part.coords) - 1

print ("Expected exploded segments:", expected_segments)

Expected exploded segments: 590355


# Figuring out why feature counts are different in exploded files 

In [2]:
old = gpd.read_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/data_for_redundancy/hawaii_roads_exploded.gpkg",
    layer="roads_exploded"
)

new = gpd.read_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/roads_exploded.gpkg",
    layer="roads_exploded"
)

In [3]:
print(len(old))
print(len(new))

589735
590355


In [7]:
comparison = pd.concat (
    [
        old["highway"].value_counts().rename("old"),
        new["highway"].value_counts().rename("new")
    ], 
    axis=1 
).fillna(0) 

comparison["difference"] = (
    comparison["new"] - comparison["old"]
) 

print(comparison.sort_index())

                     old     new  difference
highway                                     
living_street    13901.0   13909         8.0
motorway          2627.0    2628         1.0
motorway_link     4084.0    4098        14.0
primary          16527.0   16540        13.0
primary_link      3232.0    3231        -1.0
residential     157754.0  157905       151.0
road                 0.0      54        54.0
secondary        15743.0   15752         9.0
secondary_link    1638.0    1638         0.0
service         272233.0  272617       384.0
tertiary         61019.0   61043        24.0
tertiary_link     2365.0    2365         0.0
trunk             3869.0    3873         4.0
trunk_link        1441.0    1441         0.0
unclassified     33302.0   33261       -41.0


In [8]:
# checking if the osm IDs are the same in each file

old_ids = set(old["osm_id"].dropna())
new_ids = set(new["osm_id"].dropna())

print("OSM ways only in old:", len(old_ids - new_ids))
print("OSM ways only in new:", len(new_ids - old_ids))

OSM ways only in old: 4
OSM ways only in new: 101


In [9]:
print("Only in old:")
print(sorted(old_ids - new_ids)[:50])

print("\nOnly in new:")
print(sorted(new_ids - old_ids)[:50])

Only in old:
['1008030832', '1190115395', '1190115397', '802143933']

Only in new:
['1573504264', '1573504265', '1573504266', '1573504267', '1573504268', '1573504274', '1573505161', '1573509819', '1573509828', '1573509834', '1573509835', '1573509837', '1573509852', '1573509853', '1573509854', '1573532338', '1573532339', '1573532340', '1573532341', '1573678706', '1573798581', '1573798582', '1573997319', '1573997321', '1574360180', '1574360181', '1574360182', '1574360183', '1574360184', '1574360185', '1574410309', '1574410310', '1574410311', '1574410312', '1574852791', '1574852826', '1574930174', '1575088519', '1575088520', '1575088521', '1575088522', '1575324643', '1575324644', '1575387339', '1575547911', '1575547912', '1575547913', '1575547914', '1575547936', '1575547937']
